# Perform significance testing on held-out test set

## Set Up

In [ ]:
import os, sys

sys.path.append(os.path.abspath("../../"))
from src.config import BASE_PATH, SEED
from src.utils import BEST_PARAM_DICT
from src.sig import (
    prep_train_test_model,
    get_baseline_reduced,
    get_p,
    format_p_val,
    get_hyp1_null,
    get_hyp2_null,
    get_raw_p,
    plot_null_observed,
)
import pandas as pd
from sklearn.metrics import r2_score
import numpy as np

In [ ]:
all_perm_dir = BASE_PATH / "results/perm_res/preds"
train_data_dir = BASE_PATH / "data"
test_data_dir = BASE_PATH / "data/test_data/cleaned"
model_dir = BASE_PATH / "models"
result_export_path = BASE_PATH / "results/test"
SHARED_DICT = {
    "max_iter": 50000,
    "seed": SEED,
    "linear_type": "ridge",
}
START_PERM = 0
END_PERM = 1000  # non-inclusive

## Hypotheses 1A/1B:

- Hypotheses:
	- H_A: r^2(<full_model>) - r^2(<reduced_model>) > 0
	- H_0: r^2(<full_model>) - r^2(<reduced_model>) <= 0

- A/B refers to self/parent cohorts

Get null distribution for each combination of self/parent and (SQT-SQ)/(ST-S)/(QT-Q) full-reduced pairs

In [ ]:
null_dist_results_hyp1 = pd.DataFrame()
for impact_type in ["self", "parent"]:
    print(f"{impact_type}...")
    for residual_setup in ["SQT", "ST", "QT"]:
        print(f"\t{residual_setup}...")
        X_train, y_train, X_test, y_test, model = prep_train_test_model(
            selected_params=BEST_PARAM_DICT[residual_setup][impact_type],
            impact_type=impact_type,
            train_data_dir=train_data_dir,
            test_data_dir=test_data_dir,
            shared_dict=SHARED_DICT,
        )

        ## Get observed score
        model.fit(X_train, y_train, shuffle=None, perm_dir=None)

        test_pred_all = model.predict_all(X_test)
        # Reduced score (one of SQ, S, Q)
        reduced_r2_no_shuffle = get_baseline_reduced(
            residual_setup=residual_setup,
            y_true=y_test,
            pred_df=test_pred_all,
        )
        # Full score (one of SQT, ST, QT)
        full_r2_no_shuffle = r2_score(
            y_true=y_test, y_pred=test_pred_all["pred_stage3_total"]
        )
        # Test statistic (observed delta)
        observed_r2_diff = full_r2_no_shuffle - reduced_r2_no_shuffle

        diff_list = get_hyp1_null(
            X_train=X_train,
            y_train=y_train,
            X_test=X_test,
            y_test=y_test,
            model=model,
            all_perm_dir=all_perm_dir,
            reduced_r2_no_shuffle=reduced_r2_no_shuffle,
            start_perm=START_PERM,
            end_perm=END_PERM,
        )
        # Adding observed to null dist;
        # essentially = to adding 1 to numerator/denominator of p-value fraction
        null_col_name = f"{residual_setup}_{impact_type}"
        null_dist = [observed_r2_diff] + diff_list
        null_dist_results_hyp1[null_col_name] = null_dist

        ## Plot
        plot_null_observed(
            null_dist=np.array(null_dist),
            observed=observed_r2_diff,
            set_up=residual_setup,
            impact_type=impact_type,
        )

display(null_dist_results_hyp1)

Calculate p-vals from null dist w/ FWER correction

In [ ]:
p_val_res_hyp1 = []
p_val_order_dict = {}
for impact_type in ["self", "parent"]:
    print(f"{impact_type}...")
    col_SQT = f"SQT_{impact_type}"
    col_ST = f"ST_{impact_type}"
    col_QT = f"QT_{impact_type}"
    df_sub = null_dist_results_hyp1[[col_SQT, col_ST, col_QT]]

    ## WESTFALL-YOUNG
    ## Iteration 1 --> max over ALL
    p_iter1, col_max1 = get_p(
        df_sub, impact_type=impact_type, add_1_flag=False, plot_histo=True
    )
    ## Iteration 2 --> max over next 2
    df_sub = df_sub.drop(col_max1, axis=1)
    p_iter2, col_max2 = get_p(
        df_sub, impact_type=impact_type, add_1_flag=False, plot_histo=True
    )
    ## Iteration 3 --> Just remaining sub-hypothesis
    df_sub = df_sub.drop(col_max2, axis=1)
    p_iter3, col_max3 = get_p(
        df_sub, impact_type=impact_type, add_1_flag=False, plot_histo=True
    )

    p_val_res_hyp1.append(
        {
            "impact": impact_type,
            col_max1.split("_")[0]: format_p_val(p_iter1),
            col_max2.split("_")[0]: format_p_val(max([p_iter1, p_iter2])),
            col_max3.split("_")[0]: format_p_val(max([p_iter1, p_iter2, p_iter3])),
        }
    )
    p_val_order_dict[impact_type] = [col_max1, col_max2, col_max3]

p_val_df_hyp1 = pd.DataFrame(p_val_res_hyp1)
display(p_val_df_hyp1)
print(f"Order of hypotheses: {p_val_order_dict}")

## Hypotheses 2A/2B:

- Hypotheses:
	- H_A: r^2(ST) - r^2(SQ) > 0
	- H_0: r^2(ST) - r^2(SQ) <= 0

- A/B refers to self/parent cohorts

In [ ]:
p_val_dict_hyp_2 = {}
null_dist_results_hyp_2 = pd.DataFrame()
for impact_type in ["self", "parent"]:
    ## Create and train model
    X_train_SQT, y_train_SQT, X_test_SQT, y_test_SQT, model_SQT = prep_train_test_model(
        selected_params=BEST_PARAM_DICT["SQT"][impact_type],
        impact_type=impact_type,
        train_data_dir=train_data_dir,
        test_data_dir=test_data_dir,
        shared_dict=SHARED_DICT,
    )
    X_train_ST, y_train_ST, X_test_ST, y_test_ST, model_ST = prep_train_test_model(
        selected_params=BEST_PARAM_DICT["ST"][impact_type],
        impact_type=impact_type,
        train_data_dir=train_data_dir,
        test_data_dir=test_data_dir,
        shared_dict=SHARED_DICT,
    )

    assert y_train_SQT.equals(y_train_ST)
    assert y_test_SQT.equals(y_test_ST)
    assert X_train_SQT[["DAWBA_ID", "eligible_group"]].equals(
        X_train_ST[["DAWBA_ID", "eligible_group"]]
    )

    model_SQT.fit(X_train_SQT, y_train_SQT)
    model_ST.fit(X_train_ST, y_train_ST)

    #### 1) Get preds from each
    # 3-layer model- extract SQ layer preds from SQT model
    y_pred_SQ = model_SQT.predict_all(X_test_SQT)["pred_stage2_total"]
    # 2-layer model - ST is final model output here, no need to extract
    y_pred_ST = model_ST.predict(X_test_ST)

    base_r2_SQ = r2_score(y_true=y_test_SQT, y_pred=y_pred_SQ)
    base_r2_ST = r2_score(y_true=y_test_ST, y_pred=y_pred_ST)

    observed_diff = base_r2_ST - base_r2_SQ
    ### 2) FOR EACH perm-- randomly swap preds
    diff_list = get_hyp2_null(
        y_pred_SQ=y_pred_SQ,
        y_pred_ST=y_pred_ST,
        y_true=y_test_SQT,  # all y_test same, so set arbitrarily
        all_perm_dir=all_perm_dir,
        start_perm=START_PERM,
        end_perm=END_PERM,
    )
    # Adding baseline to null;
    # essentially = to adding 1 to numerator/denominator
    null_dist = [observed_diff] + diff_list
    null_dist_results_hyp_2[impact_type] = null_dist
    ### 3) Get p-val
    p_val = get_raw_p(
        null_dist=np.array(null_dist), observed=observed_diff, add_1_flag=False
    )
    p_val_dict_hyp_2[impact_type] = format_p_val(p_val)

    ## PLOT
    plot_null_observed(
        null_dist=np.array(null_dist),
        observed=observed_diff,
        set_up="(ST-QT)",
        impact_type=impact_type,
    )

print(p_val_dict_hyp_2)

## Export results

In [ ]:
final_result_df = p_val_df_hyp1.rename(
    columns={
        "ST": "ST-S",
        "SQT": "SQT-SQ",
        "QT": "QT-Q",
    }
)
final_result_df["ST-SQ"] = [p_val_dict_hyp_2["self"], p_val_dict_hyp_2["parent"]]
export_path = result_export_path / "tables/corrected_p.tsv"
if export_path.exists():
    raise FileExistsError
export_path.parent.mkdir(exist_ok=True, parents=True)
final_result_df.to_csv(export_path, sep="\t")